# 02 — Estruturação Stanford CheXpert — Center Crop
**Projeto Integrador SENAI FATESG 2025/2026**  
6 labels em comum | Center Crop → 512×512 | Target ~14.800/label | Frontais only

## SEÇÃO 0 — Setup e Configurações

In [26]:
!pip install -q Pillow tqdm pandas numpy
import os, json, time, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# Constrói índice patient → batch
import os

CHEX_DIR = '/content/drive/MyDrive/datasets_pi_2026/CheXpert_extracted'

patient_to_batch = {}

# Batches de treino — pacientes direto na raiz do batch
for batch in ['batch2_train1', 'batch3_train2', 'batch4_train3']:
    batch_path = os.path.join(CHEX_DIR, batch)
    if not os.path.exists(batch_path):
        print(f'NÃO ENCONTRADO: {batch_path}')
        continue
    for patient in os.listdir(batch_path):
        if patient.startswith('patient'):
            patient_to_batch[patient] = (batch, '')

# Batch do valid — pacientes dentro da subpasta 'valid/'
valid_path = os.path.join(CHEX_DIR, 'batch1_validate_csv', 'valid')
for patient in os.listdir(valid_path):
    if patient.startswith('patient'):
        patient_to_batch[patient] = ('batch1_validate_csv', 'valid')

print(f'Total pacientes indexados: {len(patient_to_batch):,}')

Total pacientes indexados: 64,740


In [28]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
BASE_DRIVE = '/content/drive/MyDrive'
CHEX_DIR   = f'{BASE_DRIVE}/datasets_pi_2026/CheXpert_extracted'
TRAIN_CSV  = f'{CHEX_DIR}/meta/train.csv'
VALID_CSV  = f'{CHEX_DIR}/meta/valid.csv'

OUT_DIR    = f'{BASE_DRIVE}/datasets_pi_2026/estruturacao/Stanford_Cropping'
OUT_IMAGES = f'{OUT_DIR}/images_512'
CKPT_PATH  = f'{OUT_DIR}/ckpt_estruturacao_stanford_cropping.json'

os.makedirs(OUT_IMAGES, exist_ok=True)

MODE = 'CROPPING'

# ── Labels canônicas ──────────────────────────────────────────────────────────
LABEL_MAP = {
    'Atelectasis'    : 'atelectasis',
    'Cardiomegaly'   : 'cardiomegaly',
    'Pleural Effusion': 'pleural_effusion',
    'Pneumothorax'   : 'pneumothorax',
    'Consolidation'  : 'consolidation',
    'Edema'          : 'edema',
    'No Finding'     : 'no_finding',
}
LABELS_6   = ['atelectasis','cardiomegaly','pleural_effusion',
              'pneumothorax','consolidation','edema']
ALL_LABELS = LABELS_6 + ['no_finding']

TARGET = {lbl: 14800 for lbl in ALL_LABELS}
TARGET_SIZE = (512, 512)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'Modo: {MODE} | Configurações carregadas.')

Modo: CROPPING | Configurações carregadas.


## SEÇÃO 1 — Carregamento e Mapeamento do CSV

In [29]:
# ── Lê CSVs ────────────────────────────────────────────────────────────────────
df_train_raw = pd.read_csv(TRAIN_CSV)
df_valid_raw = pd.read_csv(VALID_CSV)
print(f'Train bruto: {len(df_train_raw):,} | Valid bruto: {len(df_valid_raw):,}')
print(f'Colunas: {list(df_train_raw.columns[:10])}...')

Train bruto: 223,414 | Valid bruto: 234
Colunas: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion']...


In [30]:
# ── Filtra apenas frontais ─────────────────────────────────────────────────────
df_train_front = df_train_raw[df_train_raw['Frontal/Lateral'] == 'Frontal'].copy()
df_valid_front = df_valid_raw[df_valid_raw['Frontal/Lateral'] == 'Frontal'].copy()
print(f'Frontais train: {len(df_train_front):,} | Frontais valid: {len(df_valid_front):,}')

Frontais train: 191,027 | Frontais valid: 202


In [31]:
# ── Mapeia labels canônicas e cria colunas ─────────────────────────────────────
def map_labels(df):
    df = df.copy()
    for canon in ALL_LABELS:
        df[canon] = 0
        df[f'{canon}_uones']   = 0
        df[f'{canon}_uignore'] = np.nan

    for orig, canon in LABEL_MAP.items():
        if orig not in df.columns:
            continue
        df[f'{canon}_uones'] = df[orig].apply(
            lambda v: 1 if v == 1 or v == -1 else (0 if v == 0 else np.nan))
        df[f'{canon}_uignore'] = df[orig].apply(
            lambda v: 1 if v == 1 else (0 if v == 0 else np.nan))
        df[canon] = df[orig].apply(lambda v: 1 if v == 1 else 0)

    def build_path(p):
        parts = str(p).split('/')
        patient = parts[2]
        study   = parts[3]
        fname   = parts[4]
        entry = patient_to_batch.get(patient)
        if entry is None:
            return None
        batch, subdir = entry
        if subdir:
            return os.path.join(CHEX_DIR, batch, subdir, patient, study, fname)
        else:
            return os.path.join(CHEX_DIR, batch, patient, study, fname)

    df['abs_path'] = df['Path'].apply(build_path)

    n_none = df['abs_path'].isna().sum()
    if n_none > 0:
        print(f'{n_none:,} paths não mapeados')

    return df

df_train = map_labels(df_train_front)
df_valid = map_labels(df_valid_front)

# ── Filtra imagens relevantes ─────────────────────────────────────────────────
mask_train = df_train[ALL_LABELS].sum(axis=1) > 0
df_train = df_train[mask_train].copy().reset_index(drop=True)
mask_valid = df_valid[ALL_LABELS].sum(axis=1) > 0
df_valid = df_valid[mask_valid].copy().reset_index(drop=True)

print(f'Train filtrado: {len(df_train):,} | Valid filtrado: {len(df_valid):,}')
print('\nDistribuição por label (+1 confirmados):')
for lbl in ALL_LABELS:
    n = df_train[lbl].sum()
    print(f'  {lbl:<20}: {n:>8,} ({n/len(df_train)*100:.2f}%)')

Train filtrado: 149,569 | Valid filtrado: 159

Distribuição por label (+1 confirmados):
  atelectasis         :   29,720 (19.87%)
  cardiomegaly        :   23,385 (15.63%)
  pleural_effusion    :   76,899 (51.41%)
  pneumothorax        :   17,693 (11.83%)
  consolidation       :   12,983 (8.68%)
  edema               :   49,675 (33.21%)
  no_finding          :   16,974 (11.35%)


## SEÇÃO 2 — Estratégia de Balanceamento

In [32]:
# ── Undersample todos os labels para ~14.800 ─────────────────────────────────
random.seed(RANDOM_SEED)
selected_idx = set()

for lbl in ALL_LABELS:
    positives = df_train[df_train[lbl] == 1].index.tolist()
    target = TARGET[lbl]
    if len(positives) <= target:
        selected_idx.update(positives)
        print(f'  {lbl:<20}: {len(positives):>8,} → mantém todos (abaixo do target)')
    else:
        sampled = random.sample(positives, target)
        selected_idx.update(sampled)
        print(f'  {lbl:<20}: {len(positives):>8,} → undersample → {target:,}')

df_balanced = df_train.loc[sorted(selected_idx)].copy().reset_index(drop=True)
print(f'\nTotal selecionado: {len(df_balanced):,} imagens')
print('\nDistribuição pós-balanceamento:')
for lbl in ALL_LABELS:
    n = df_balanced[lbl].sum()
    print(f'  {lbl:<20}: {n:>8,}')

  atelectasis         :   29,720 → undersample → 14,800
  cardiomegaly        :   23,385 → undersample → 14,800
  pleural_effusion    :   76,899 → undersample → 14,800
  pneumothorax        :   17,693 → undersample → 14,800
  consolidation       :   12,983 → mantém todos (abaixo do target)
  edema               :   49,675 → undersample → 14,800
  no_finding          :   16,974 → undersample → 14,800

Total selecionado: 87,258 imagens

Distribuição pós-balanceamento:
  atelectasis         :   19,865
  cardiomegaly        :   17,504
  pleural_effusion    :   37,752
  pneumothorax        :   15,410
  consolidation       :   12,983
  edema               :   27,102
  no_finding          :   14,800


## SEÇÃO 3 — Função de Preprocessing (Center Crop)

In [33]:
from PIL import Image

def process_image_cropping(src_path, out_path, size=(512,512)):
    """
    Center Crop: carrega JPEG, aplica Center Crop
    para quadrado, resize para 512×512, salva PNG.
    """
    with Image.open(src_path) as img:
        img = img.convert('L')  # grayscale
        w, h = img.size
        # Center Crop: corta o lado maior para igualar ao menor
        min_dim = min(w, h)
        left = (w - min_dim) // 2
        top  = (h - min_dim) // 2
        img  = img.crop((left, top, left + min_dim, top + min_dim))


        img = img.resize(size, Image.LANCZOS)
        img.save(out_path, format='PNG')

print(f'Função process_image_cropping definida.')

Função process_image_cropping definida.


## SEÇÃO 4 — Processamento das Imagens (com Checkpoint)

In [34]:
# ── Carrega checkpoint ───────────────────────────────────────────────────────
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        ckpt = json.load(f)
    done = set(ckpt.get('done', []))
    print(f'Checkpoint: {len(done):,} imagens concluídas')
else:
    done = set()
    print('Sem checkpoint — iniciando do zero.')
errors = []

Checkpoint: 87,258 imagens concluídas


In [35]:
# ── Processa train balanceado ────────────────────────────────────────────────
rows_train = []
t0 = time.time()
process_fn = process_image_cropping

for i, row in tqdm(df_balanced.iterrows(), total=len(df_balanced), desc='Train'):
    src = row['abs_path']
    img_key = row['Path']

    # Nome de saída: substitui separadores por _ para achatar estrutura
    out_name = str(row['Path']).replace('/', '_').replace('CheXpert-v1.0-small_', '')
    out_name = os.path.splitext(out_name)[0] + '.png'  # remove extensão original antes de adicionar .png
    out_path = os.path.join(OUT_IMAGES, out_name)

    row_data = {'img_path': out_path, 'patient_id': str(row['Path']).split('/')[2]}
    for lbl in ALL_LABELS:
        row_data[lbl]               = int(row[lbl])
        row_data[f'{lbl}_uones']  = row[f'{lbl}_uones']
        row_data[f'{lbl}_uignore']= row[f'{lbl}_uignore']

    if img_key in done:
        rows_train.append(row_data)
        continue

    if not os.path.exists(src):
        errors.append(f'NOT_FOUND: {src}')
        continue

    try:
        process_fn(src, out_path)
        done.add(img_key)
        rows_train.append(row_data)
    except Exception as e:
        errors.append(f'{img_key}: {e}')

    if len(done) % 500 == 0:
        with open(CKPT_PATH, 'w') as f:
            json.dump({'done': list(done)}, f)

print(f'Train: {len(rows_train):,} | Erros: {len(errors):,} | Tempo: {time.time()-t0:.0f}s')

Train: 100%|██████████| 87258/87258 [00:09<00:00, 8945.69it/s]

Train: 87,258 | Erros: 0 | Tempo: 10s


In [36]:
# ── Processa valid ───────────────────────────────────────────────────────────
rows_valid = []

for i, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc='Valid'):
    src = row['abs_path']
    out_name = str(row['Path']).replace('/', '_').replace('CheXpert-v1.0-small_', '')
    out_name = os.path.splitext(out_name)[0] + '.png'  # remove extensão original antes de adicionar .png
    out_path = os.path.join(OUT_IMAGES, out_name)

    row_data = {'img_path': out_path, 'patient_id': str(row['Path']).split('/')[2]}
    for lbl in ALL_LABELS:
        row_data[lbl]               = int(row[lbl])
        row_data[f'{lbl}_uones']  = row[f'{lbl}_uones']
        row_data[f'{lbl}_uignore']= row[f'{lbl}_uignore']

    if not os.path.exists(src):
        errors.append(f'NOT_FOUND_VALID: {src}')
        continue
    try:
        process_fn(src, out_path)
        rows_valid.append(row_data)
    except Exception as e:
        errors.append(f'VALID_{i}: {e}')

print(f'Valid: {len(rows_valid):,} | Erros acumulados: {len(errors):,}')

Valid: 100%|██████████| 159/159 [03:51<00:00,  1.46s/it]

Valid: 159 | Erros acumulados: 0


## SEÇÃO 5 — Cálculo de Pesos e Geração dos CSVs

In [37]:
# ── Monta DataFrames ─────────────────────────────────────────────────────────
df_out_train = pd.DataFrame(rows_train)
df_out_valid = pd.DataFrame(rows_valid)
N = len(df_out_train)

# ── peso_uones: calculado sobre (+1 + -1) convertidos ─────────────────────────
label_weights_uones = {}
for lbl in ALL_LABELS:
    col = f'{lbl}_uones'
    n_pos = (df_out_train[col] == 1).sum()
    label_weights_uones[lbl] = N / (len(ALL_LABELS) * n_pos) if n_pos > 0 else 1.0

# ── peso_uignore: calculado sobre (+1) apenas ─────────────────────────────────
label_weights_uignore = {}
for lbl in ALL_LABELS:
    n_pos = df_out_train[lbl].sum()
    label_weights_uignore[lbl] = N / (len(ALL_LABELS) * n_pos) if n_pos > 0 else 1.0

def calc_peso(row, weights, suffix=''):
    pos = [lbl for lbl in ALL_LABELS
           if (row[f'{lbl}{suffix}'] == 1 if suffix else row[lbl] == 1)]
    if not pos:
        return 1.0
    return float(np.mean([weights[lbl] for lbl in pos]))

df_out_train['peso_uones']   = df_out_train.apply(
    lambda r: calc_peso(r, label_weights_uones, '_uones'), axis=1)
df_out_train['peso_uignore'] = df_out_train.apply(
    lambda r: calc_peso(r, label_weights_uignore), axis=1)
df_out_valid['peso_uones']   = df_out_valid.apply(
    lambda r: calc_peso(r, label_weights_uones, '_uones'), axis=1)
df_out_valid['peso_uignore'] = df_out_valid.apply(
    lambda r: calc_peso(r, label_weights_uignore), axis=1)

print('Pesos calculados.')
print('\npeso_uones label weights:')
for lbl, w in label_weights_uones.items():
    print(f'  {lbl:<20}: {w:.4f}')
print('\npeso_uignore label weights:')
for lbl, w in label_weights_uignore.items():
    print(f'  {lbl:<20}: {w:.4f}')

Pesos calculados.

peso_uones label weights:
  atelectasis         : 0.4273
  cardiomegaly        : 0.6407
  pleural_effusion    : 0.3081
  pneumothorax        : 0.7780
  consolidation       : 0.6258
  edema               : 0.4131
  no_finding          : 0.8423

peso_uignore label weights:
  atelectasis         : 0.6275
  cardiomegaly        : 0.7121
  pleural_effusion    : 0.3302
  pneumothorax        : 0.8089
  consolidation       : 0.9601
  edema               : 0.4599
  no_finding          : 0.8423


In [38]:
# ── Salva CSVs ────────────────────────────────────────────────────────────────
cols_base = ['img_path', 'patient_id'] + ALL_LABELS
cols_uones   = [f'{lbl}_uones'   for lbl in ALL_LABELS]
cols_uignore = [f'{lbl}_uignore' for lbl in ALL_LABELS]
cols_pesos   = ['peso_uones', 'peso_uignore']
cols_final   = cols_base + cols_uones + cols_uignore + cols_pesos

csv_train = f'{OUT_DIR}/chex_crop_train.csv'
csv_valid = f'{OUT_DIR}/chex_crop_valid.csv'

df_out_train[cols_final].to_csv(csv_train, index=False)
df_out_valid[cols_final].to_csv(csv_valid, index=False)

print(f'CSVs salvos:')
print(f'  {csv_train}  ({len(df_out_train):,} linhas)')
print(f'  {csv_valid}  ({len(df_out_valid):,} linhas)')
print(f'\nErros totais: {len(errors):,}')
if errors[:5]:
    print('Primeiros erros:', errors[:5])

with open(CKPT_PATH, 'w') as f:
    json.dump({'done': list(done), 'completed': True}, f)
print('\n Estruturação Stanford Center Crop concluída.')

CSVs salvos:
  /content/drive/MyDrive/datasets_pi_2026/estruturacao/Stanford_Cropping/chex_crop_train.csv  (87,258 linhas)
  /content/drive/MyDrive/datasets_pi_2026/estruturacao/Stanford_Cropping/chex_crop_valid.csv  (159 linhas)

Erros totais: 0

 Estruturação Stanford Center Crop concluída.


## SEÇÃO 6 — Renomeação
Problema pontual, já arrumado no corpo do código principal!

In [39]:
import os
for fname in os.listdir(OUT_IMAGES):
    if fname.endswith('.jpg.png'):
        old = os.path.join(OUT_IMAGES, fname)
        new = os.path.join(OUT_IMAGES, fname.replace('.jpg.png', '.png'))
        os.rename(old, new)

print('Renomeação concluída.')

Renomeação concluída.


In [40]:
import os
sample = os.listdir(f'{OUT_IMAGES}')[:3]
print(sample)

['CheXpert-v1.0_train_patient62460_study1_view1_frontal.png', 'CheXpert-v1.0_train_patient62469_study1_view1_frontal.png', 'CheXpert-v1.0_train_patient62469_study2_view1_frontal.png']
